In [ ]:
# !git clone https://github.com/vuthetam/mscoco_image_captioning.git

# import os
# import sys
# sys.path.append("/kaggle/working/mscoco_image_captioning")

# !pip install pycocoevalcap

In [ ]:
import os
import pandas as pd
import torch

from config import Backbone
from dataset import create_img_transform
from model.decoder import CaptionDecoder
from model.encoder import create_encoder
from vocabulary import Vocabulary
from checkpoint import load_checkpoint
from evaluation import evaluate_caption_metrics


BACKBONE: Backbone = "clip_vit_b16"

DATASET_COCO_PATH = "dataset/dataset_coco.json"
IMAGES_DIR = "dataset/images"
LOAD_CHECKPOINT_DIR = "checkpoints/" + BACKBONE
# DATASET_COCO_PATH = "/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json"
# IMAGES_DIR = "/kaggle/input/datasets/vuthetam/mscoco-2014/images"
# LOAD_CHECKPOINT_DIR = "/kaggle/input/notebooks/vuthetam/mscoco-clip-vit-b16/checkpoints/" + BACKBONE

LOAD_BEST_CHECKPOINT_PATH = os.path.join(LOAD_CHECKPOINT_DIR, "best_checkpoint.pt")

BATCH_SIZE = 32
MIN_FREQ = 5
NUM_EPOCHS = 10

D_MODEL = 512
NHEAD = 8
NUM_LAYERS = 4
MAX_LEN = 32
DROPOUT = 0.1

LEARNING_RATE = 1e-4

BEAM_SIZE = 5
LENGTH_PENALTY = 0.7



In [ ]:
df = pd.read_json(DATASET_COCO_PATH)
images_df = pd.json_normalize(df["images"])
test_df = images_df[images_df["split"] == "test"]

train_df = pd.json_normalize(df["images"], record_path="sentences", meta=["split"])
train_df = train_df[train_df["split"].isin(["train", "restval"])]

tokens_list = train_df["tokens"].tolist()
vocab = Vocabulary(MIN_FREQ)
vocab.build_from_tokens(tokens_list)

img_transform = create_img_transform(BACKBONE)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = create_encoder(BACKBONE, D_MODEL)
decoder = CaptionDecoder(len(vocab), D_MODEL, MAX_LEN, NHEAD, DROPOUT, NUM_LAYERS)

encoder.to(device)
decoder.to(device)

if os.path.isfile(LOAD_BEST_CHECKPOINT_PATH):
    epoch, _train_loss, best_val_loss = load_checkpoint(LOAD_BEST_CHECKPOINT_PATH, encoder, decoder, device)
    print(f"Loaded checkpoint | Epoch {epoch} | Val loss: {best_val_loss:.4f}")

predictions, references, metric_scores = evaluate_caption_metrics(encoder, decoder, vocab, test_df, IMAGES_DIR, img_transform, MAX_LEN, BEAM_SIZE, LENGTH_PENALTY)

for metric, score in metric_scores.items():
    print(f"{metric}: {score}")

In [ ]:
from PIL import Image
from IPython.display import display

def show_captions_samples(predictions, references, df, images_dir, num_samples=5, indices=None):
    evaluated_filename = list(predictions.keys())

    if indices is None:
        filenames = evaluated_filename[:num_samples]
    else:
        filenames = [evaluated_filename[idx] for idx in indices]

    for filename in filenames:
        row = test_df[test_df["filename"] == filename].iloc[0]

        image_path = os.path.join(images_dir, row["filepath"], row["filename"])
        with Image.open(image_path) as img:
            image = img.convert("RGB")

        print(f"File name: {filename}\nPrediction: {predictions[filename]}\nReferences:")
        for ground_truth in references[filename]:
            print(f"- {ground_truth}")
        display(image)
        print("-"*100)


show_captions_samples(predictions, references, test_df, IMAGES_DIR, indices=[3,8,0,4,7])